# GOLD ATP RANKING

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("fact_player_ranking").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [77]:
# # silver
# tb_atp_rankings = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "silver.tb_atp_rankings")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

# # gold
# tb_date = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "gold.dim_date")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

# tb_players = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "gold.dim_players")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

tb_atp_rankings = spark.read.csv("../../../data/silver/tb_atp_rankings.csv", sep=',', header=True)
tb_date = spark.read.csv("../../../data/gold/dimension/dim_date.csv", sep=',', header=True)
tb_players = spark.read.csv("../../../data/gold/dimension/dim_players.csv", sep=',', header=True)

## Ranking

In [107]:
df = (
    tb_atp_rankings.alias("r")
    .join(
        tb_date.alias("d"),
        f.col("r.DATE_WEEK_RANKING") == f.col("d.DATE"),
        'right'
    )
    .join(
        tb_players.alias("p"),
        f.initcap(f.trim(f.regexp_replace(f.col("r.DES_PLAYER_NAME"), "-", " "))) == f.col("p.DES_PLAYER_NAME"),
        'right'
    )

    .select(
        f.col("d.DATE").alias("DATE_WEEK_RANKING"),
        "p.DES_PLAYER_NAME",
        "r.NUM_PLAYER_RANK",
        "r.NUM_PLAYER_RANK_PTS",
        "r.NUM_PLAYER_LE_PTS",
        "r.NUM_PLAYER_DROP_PTS",
        "r.NUM_PLAYER_NEXT_BEST",
        "p.DATE_INGESTION",
    )
)

## Save dataframe

### Local

In [108]:
df.toPandas().to_csv(
    r"../../../data/gold/fact/fact_player_ranking.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [109]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.fact_player_ranking")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)